# **Pre-Trained Model 1 - Efficient Net B0**

EfficientNetB0 was selected as the first pretrained architecture based on three
key criteria: parameter efficiency, proven medical imaging performance and
architectural design alignment with our task.


### Imports

In [7]:
import os
import sys
import keras.applications
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

if os.getcwd().endswith('models'):
    os.chdir('..')
    
from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

### **Data** Configuration

In [2]:
# path for the new images
base_path = "./data" 
aug_dir = os.path.join(base_path, "HAM10000_augmented")
if not os.path.exists(aug_dir):
    os.makedirs(aug_dir)

In [12]:
# CORRIGIR COM BASE NO AUGMENTED

train_df = pd.read_csv('data/augmented_metadata.csv')
val_df   = pd.read_csv('data/val_split.csv')
test_df  = pd.read_csv('data/test_split.csv')

# 3. Clean up the columns to avoid duplicates
for df in [train_df, val_df, test_df]:
    # Drop the original 'image_path' if 'cleaned_path' exists to avoid duplicates
    if 'cleaned_path' in df.columns and 'image_path' in df.columns:
        df.drop(columns=['image_path'], inplace=True)
    
    # Now rename safely
    df.rename(columns={'cleaned_path': 'image_path'}, inplace=True)
    
    # Add encoded labels BEFORE calling make_dataset
    df['dx_encoded'] = df['dx'].map(label2idx).astype(int)
    df['dataset'] = 'original'


BATCH_SIZE = 32 # VERIFICAR PARA QUE É QUE ISTO SERVE

#train_ds = make_dataset(train_df, resize_function=format_area_matched, shuffle=True, repeat=True)  
#val_ds   = make_dataset(val_df, resize_function=format_area_matched)                               
#test_ds  = make_dataset(test_df, resize_function=format_area_matched)

train_ds = make_dataset(train_df, shuffle=True, repeat=True)  
val_ds   = make_dataset(val_df)                               
test_ds  = make_dataset(test_df)

train_df['dataset'] = 'original'
val_df['dataset'] = 'original'
test_df['dataset'] = 'original'

# Add encoded labels
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

for df in [train_df, val_df, test_df]:
    df['dx_encoded'] = df['dx'].map(label2idx)

N_CLASSES  = len(label2idx) # VERIFICAR PARA QUE É QUE ISTO SERVE

In [13]:
STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

Steps per epoch: 444
Class weights: {0: 1.6051948051948053, 1: 1.2503518648838845, 2: 0.7744360902255639, 3: 2.9861344537815127, 4: 0.7824938067712635, 5: 0.4512380952380952, 6: 2.188115763546798}


### **Model** Configuration

In [ ]:
UNFREEZE_FROM_B = -30   # unfreeze last 30 layers of EfficientNetB0

def build_efficientnet():
    base = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling=None
        
    )
    base.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))

    scaled_inputs = keras.layers.Rescaling(scale=255.0)(inputs)

    x = base(scaled_inputs, training=False)

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.4)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(N_CLASSES)(x)
    return keras.Model(inputs, outputs, name="efficientnet_b0")

In [ ]:
# phase 1: head only
model_pt1 = build_efficientnet() # Model Pre-Trained 1
model_pt1.summary()



model_pt1.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history1_pt1 = model_pt1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_b_phase1.weights.h5",
                            patience_es=6, patience_lr=3)
)

plot_history(history1_pt1, "Model B — EfficientNetB0 Phase 1 (head only)")

Epoch 1/20
444/444 [==============================] - ETA: 0s - loss: 1.6571 - accuracy: 0.3059
Epoch 1: val_loss improved from inf to 6.98626, saving model to checkpoints\model_b_phase1.weights.h5
444/444 [==============================] - 137s 247ms/step - loss: 1.6571 - accuracy: 0.3059 - val_loss: 6.9863 - val_accuracy: 0.0366 - lr: 0.0010
Epoch 2/20
444/444 [==============================] - ETA: 0s - loss: 1.3281 - accuracy: 0.4456
Epoch 2: val_loss improved from 6.98626 to 2.08282, saving model to checkpoints\model_b_phase1.weights.h5
444/444 [==============================] - 192s 433ms/step - loss: 1.3281 - accuracy: 0.4456 - val_loss: 2.0828 - val_accuracy: 0.2179 - lr: 0.0010
Epoch 3/20
152/444 [=========>....................] - ETA: 2:19 - loss: 1.3281 - accuracy: 0.4560

In [ ]:
# Phase 2: Fine-tune top layers
base_b = model_pt1.layers[1]
base_b.trainable = True
for layer in base_b.layers[:UNFREEZE_FROM_B]:
    layer.trainable = False

model_pt1.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history2_b0 = model_pt1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_b_best.weights.h5",
                            patience_es=8, patience_lr=4)
)

plot_history(history2_pt1, "Model B — EfficientNetB0 Phase 2 (fine-tune)")
results_b = evaluate_model(model_pt1, test_ds, test_df, label2idx, "Model B — EfficientNetB0")
